In [7]:
import pandas as pd
import numpy as np
import re
from typing import List, Dict, Set, Tuple
from pathlib import Path
import json

In [8]:

class RegulationFilter:
    """Filter LLM outputs to extract only genuine regulatory constraints and requirements."""
    
    def __init__(self):
        self.df = None
        self.filtered_regulations = None
        
        # Define regulatory keywords that indicate actual constraints
        self.regulatory_keywords = {
            'mandatory_verbs': [
                'must', 'shall', 'required', 'mandatory', 'prohibited', 'forbidden',
                'restricted', 'limited', 'constrained', 'regulated', 'controlled',
                'comply', 'accordance', 'pursuant', 'accordance with', 'subject to',
                'ensure', 'maintain', 'demonstrate', 'provide', 'submit', 'obtain',
                'authorize', 'approve', 'permit', 'license', 'certify', 'verify'
            ],
            'regulatory_phrases': [
                'not exceed', 'minimum distance', 'maximum distance', 'within', 'outside',
                'at least', 'no more than', 'greater than', 'less than', 'equal to',
                'radius of', 'buffer zone', 'setback', 'clearance', 'spacing',
                'mitigation measures', 'protective measures', 'safety measures',
                'environmental protection', 'marine protected', 'wildlife protection',
                'noise limits', 'vibration limits', 'electromagnetic', 'interference',
                'exclusion zone', 'restricted area', 'prohibited area', 'closure',
                'monitoring required', 'assessment required', 'study required',
                'consultation required', 'approval required', 'permit required'
            ],
            'constraint_types': [
                'setback', 'buffer', 'exclusion', 'restriction', 'limitation',
                'requirement', 'standard', 'guideline', 'threshold', 'limit',
                'boundary', 'zone', 'area', 'corridor', 'protection', 'mitigation',
                'monitoring', 'assessment', 'compliance', 'certification'
            ],
            'regulatory_contexts': [
                'planning', 'construction', 'operation', 'maintenance', 'decommissioning',
                'environmental', 'safety', 'navigation', 'aviation', 'fishing',
                'cultural', 'archaeological', 'wildlife', 'marine', 'coastal',
                'seabed', 'foundation', 'cable', 'turbine', 'substation'
            ]
        }
        
        # Define non-regulatory patterns that indicate descriptions
        self.non_regulatory_patterns = [
            # Technical specifications without constraints
            r'^(?:The\s+)?(?:wind\s+power\s+density|power\s+density|capacity\s+factor|wind\s+speed)',
            r'^(?:The\s+)?(?:size\s+of|diameter\s+of|height\s+of|length\s+of|weight\s+of)',
            r'^(?:The\s+)?(?:turbine|generator|rotor|blade|tower|foundation)\s+(?:has|is|would\s+be)',
            r'^(?:The\s+)?(?:project|development|facility|installation)\s+(?:consists|includes|comprises)',
            r'^(?:The\s+)?(?:cable|transmission|export)\s+(?:system|infrastructure)\s+(?:consists|includes)',
            
            # Simple factual statements
            r'^(?:The\s+)?(?:area|region|location|site)\s+(?:is|has|contains)',
            r'^(?:The\s+)?(?:water\s+depth|sea\s+depth|depth)\s+(?:is|ranges|varies)',
            r'^(?:The\s+)?(?:distance|spacing|separation)\s+(?:is|would\s+be)',
            r'^(?:There\s+(?:are|is|will\s+be))',
            
            # Descriptions of existing conditions
            r'^(?:The\s+)?(?:existing|current|present)\s+(?:conditions|infrastructure|environment)',
            r'^(?:The\s+)?(?:proposed|planned|intended)\s+(?:layout|design|configuration)',
            r'^(?:Wind\s+(?:resources|conditions|measurements))',
            r'^(?:Sea\s+(?:conditions|state|environment))',
            
            # Technical specifications
            r'^\d+\s*(?:MW|kW|kV|m|km|Hz|dB)',
            r'^\d+(?:\.\d+)?\s*(?:m/s|knots|mph)',
            r'W/m\[?\d?\]?',
            r'about\s+\d+%\s+(?:larger|smaller|more|less)',
            
            # Comparative statements
            r'compared\s+to',
            r'in\s+comparison',
            r'relative\s+to',
            r'versus',
            r'as\s+opposed\s+to'
        ]
    
    def load_data(self, csv_path: str) -> bool:
        """Load the combined regulatory constraints CSV file."""
        try:
            self.df = pd.read_csv(csv_path)
            print(f"✓ Loaded {len(self.df)} rows from {csv_path}")
            print(f"  Columns: {list(self.df.columns)}")
            return True
        except Exception as e:
            print(f"❌ Error loading data: {e}")
            return False
    
    def is_regulatory_constraint(self, text: str) -> Tuple[bool, str, float]:
        """
        Determine if a text represents a genuine regulatory constraint.
        
        Returns:
            Tuple of (is_regulatory, reason, confidence_score)
        """
        if pd.isna(text) or not isinstance(text, str) or len(text.strip()) < 10:
            return False, "Too short or empty", 0.0
        
        text_lower = text.lower().strip()
        
        # First check for non-regulatory patterns
        for pattern in self.non_regulatory_patterns:
            if re.search(pattern, text_lower):
                return False, f"Matches non-regulatory pattern: {pattern[:50]}...", 0.1
        
        # Calculate regulatory score
        score = 0.0
        reasons = []
        
        # Check for mandatory verbs (high weight)
        mandatory_count = sum(1 for verb in self.regulatory_keywords['mandatory_verbs'] 
                             if verb in text_lower)
        if mandatory_count > 0:
            score += mandatory_count * 0.3
            reasons.append(f"Contains {mandatory_count} mandatory verb(s)")
        
        # Check for regulatory phrases (high weight)
        phrase_count = sum(1 for phrase in self.regulatory_keywords['regulatory_phrases'] 
                          if phrase in text_lower)
        if phrase_count > 0:
            score += phrase_count * 0.25
            reasons.append(f"Contains {phrase_count} regulatory phrase(s)")
        
        # Check for constraint types (medium weight)
        constraint_count = sum(1 for ctype in self.regulatory_keywords['constraint_types'] 
                              if ctype in text_lower)
        if constraint_count > 0:
            score += constraint_count * 0.15
            reasons.append(f"Contains {constraint_count} constraint type(s)")
        
        # Check for regulatory contexts (low weight)
        context_count = sum(1 for context in self.regulatory_keywords['regulatory_contexts'] 
                           if context in text_lower)
        if context_count > 0:
            score += context_count * 0.1
            reasons.append(f"Contains {context_count} regulatory context(s)")
        
        # Check for numerical constraints (bonus points)
        numerical_patterns = [
            r'\d+\s*(?:km|m|miles?|feet?)\s*(?:radius|distance|buffer|setback)',
            r'(?:minimum|maximum|at\s+least|no\s+more\s+than|less\s+than|greater\s+than)\s+\d+',
            r'\d+\s*(?:%|percent)\s*(?:of|limit|threshold)',
            r'within\s+\d+',
            r'exceed\s+\d+',
            r'between\s+\d+\s+and\s+\d+'
        ]
        
        numerical_constraint_count = sum(1 for pattern in numerical_patterns 
                                       if re.search(pattern, text_lower))
        if numerical_constraint_count > 0:
            score += numerical_constraint_count * 0.2
            reasons.append(f"Contains {numerical_constraint_count} numerical constraint(s)")
        
        # Check sentence structure for regulatory language
        if re.search(r'(?:shall|must|required?\s+to)\s+(?:not\s+)?(?:exceed|be|maintain|ensure|provide)', text_lower):
            score += 0.3
            reasons.append("Uses regulatory sentence structure")
        
        # Penalty for descriptive language
        descriptive_patterns = [
            r'would\s+be',
            r'consists?\s+of',
            r'includes?',
            r'comprises?',
            r'is\s+about',
            r'approximately',
            r'roughly'
        ]
        
        descriptive_count = sum(1 for pattern in descriptive_patterns 
                               if re.search(pattern, text_lower))
        if descriptive_count > 0:
            score -= descriptive_count * 0.1
            reasons.append(f"Contains {descriptive_count} descriptive phrase(s) (penalty)")
        
        # Determine if it's regulatory based on score
        is_regulatory = score >= 0.5
        reason = "; ".join(reasons) if reasons else "No regulatory indicators found"
        
        return is_regulatory, reason, min(score, 1.0)
    
    def analyze_requirement_field(self, text: str) -> Dict:
        """Analyze the requirement field for regulatory content."""
        is_reg, reason, score = self.is_regulatory_constraint(text)
        
        return {
            'is_regulatory': is_reg,
            'confidence_score': score,
            'analysis_reason': reason,
            'text_length': len(str(text)) if pd.notna(text) else 0,
            'has_numerical_constraint': bool(re.search(r'\d+', str(text))) if pd.notna(text) else False
        }
    
    def filter_regulations(self, min_confidence: float = 0.5) -> pd.DataFrame:
        """Filter the dataset to keep only genuine regulatory constraints."""
        if self.df is None:
            print("❌ No data loaded. Call load_data() first.")
            return pd.DataFrame()
        
        print(f"🔍 Analyzing {len(self.df)} constraints for regulatory content...")
        
        # Analyze each description
        analysis_results = []
        for idx, row in self.df.iterrows():
            if idx % 100 == 0:
                print(f"  Processing row {idx}/{len(self.df)}...")
            
            description_text = row.get('description', '')
            analysis = self.analyze_requirement_field(description_text)
            analysis['index'] = idx
            analysis_results.append(analysis)
        
        # Create analysis DataFrame
        analysis_df = pd.DataFrame(analysis_results)
        
        # Merge with original data
        enhanced_df = self.df.copy()
        for col in analysis_df.columns:
            if col != 'index':
                enhanced_df[col] = analysis_df[col]
        
        # Filter for regulatory constraints
        regulatory_df = enhanced_df[
            (enhanced_df['is_regulatory'] == True) & 
            (enhanced_df['confidence_score'] >= min_confidence)
        ].copy()
        
        # Sort by confidence score (highest first)
        regulatory_df = regulatory_df.sort_values('confidence_score', ascending=False)
        
        self.filtered_regulations = regulatory_df
        
        print(f"✓ Analysis complete!")
        print(f"  Total constraints analyzed: {len(self.df)}")
        print(f"  Regulatory constraints found: {len(regulatory_df)}")
        print(f"  Filter efficiency: {len(regulatory_df)/len(self.df)*100:.1f}%")
        print(f"  Confidence range: {regulatory_df['confidence_score'].min():.3f} - {regulatory_df['confidence_score'].max():.3f}")
        
        return regulatory_df
    
    def analyze_results(self, filtered_df: pd.DataFrame):
        """Analyze and display filtering results."""
        if filtered_df.empty:
            print("❌ No regulatory constraints found")
            return
        
        print("\n" + "="*60)
        print("📊 REGULATORY CONSTRAINT ANALYSIS")
        print("="*60)
        
        # Confidence score distribution
        print(f"\n📈 CONFIDENCE SCORE DISTRIBUTION:")
        print(f"  Mean confidence: {filtered_df['confidence_score'].mean():.3f}")
        print(f"  Median confidence: {filtered_df['confidence_score'].median():.3f}")
        print(f"  Standard deviation: {filtered_df['confidence_score'].std():.3f}")
        
        # Confidence score bands
        high_confidence = filtered_df[filtered_df['confidence_score'] >= 0.8]
        medium_confidence = filtered_df[(filtered_df['confidence_score'] >= 0.6) & (filtered_df['confidence_score'] < 0.8)]
        low_confidence = filtered_df[(filtered_df['confidence_score'] >= 0.5) & (filtered_df['confidence_score'] < 0.6)]
        
        print(f"\n📊 CONFIDENCE BANDS:")
        print(f"  High confidence (≥0.8): {len(high_confidence)} constraints")
        print(f"  Medium confidence (0.6-0.8): {len(medium_confidence)} constraints")
        print(f"  Low confidence (0.5-0.6): {len(low_confidence)} constraints")
        
        # Constraint types
        if 'category' in filtered_df.columns:
            print(f"\n📋 CONSTRAINT CATEGORY DISTRIBUTION:")
            type_counts = filtered_df['category'].value_counts()
            for ctype, count in type_counts.head(10).items():
                print(f"  {ctype}: {count}")
        
        # Top regulatory constraints
        print(f"\n🏆 TOP 10 REGULATORY CONSTRAINTS:")
        top_10 = filtered_df.head(10)
        for i, (idx, row) in enumerate(top_10.iterrows(), 1):
            print(f"\n  #{i} (Confidence: {row['confidence_score']:.3f})")
            print(f"    Category: {row.get('category', 'N/A')}")
            print(f"    Description: {str(row.get('description', ''))[:150]}...")
            print(f"    Analysis: {row.get('analysis_reason', 'N/A')[:100]}...")
            print(f"    Source: {str(row.get('source_section_number', ''))[:80]}...")
    
    def save_filtered_regulations(self, filtered_df: pd.DataFrame, output_path: str = None):
        """Save filtered regulatory constraints, keeping only columns up to 'component_name'."""
        if filtered_df.empty:
            print("❌ No regulatory constraints to save")
            return
        
        if output_path is None:
            output_path = "/Users/li/Library/CloudStorage/OneDrive-UniversityofMaryland/PhD research/Python/LLM/Extract_regulations/Filtered_regulations/genuine_regulations"
        
        # Clean the output path - remove .csv extension if present to avoid double extension
        if output_path.endswith('.csv'):
            output_path = output_path[:-4]
        
        # Keep only columns up to and including 'component_name'
        original_columns = filtered_df.columns.tolist()
        if 'component_name' in original_columns:
            # Find the index of 'component_name' column
            component_name_idx = original_columns.index('component_name')
            # Keep columns from start up to and including 'component_name'
            columns_to_keep = original_columns[:component_name_idx + 1]
            filtered_output_df = filtered_df[columns_to_keep].copy()
            print(f"📋 Keeping columns: {columns_to_keep}")
        else:
            # If 'component_name' not found, keep original structure
            filtered_output_df = filtered_df.copy()
            print("⚠️  'component_name' column not found, keeping all columns")
        
        # Save as CSV
        csv_path = f"{output_path}.csv"
        filtered_output_df.to_csv(csv_path, index=False)
        print(f"💾 Filtered regulations saved to: {csv_path}")
        
        # Save high-confidence subset
        high_confidence_df = filtered_df[filtered_df['confidence_score'] >= 0.8]
        if not high_confidence_df.empty:
            # Apply same column filtering to high confidence subset
            if 'component_name' in high_confidence_df.columns:
                high_confidence_output_df = high_confidence_df[columns_to_keep].copy()
            else:
                high_confidence_output_df = high_confidence_df.copy()
            
            high_conf_path = f"{output_path}_high_confidence.csv"
            high_confidence_output_df.to_csv(high_conf_path, index=False)
            print(f"💾 High-confidence regulations saved to: {high_conf_path}")
        
        # Create summary report
        summary = {
            'total_original_constraints': len(self.df) if self.df is not None else 0,
            'regulatory_constraints': len(filtered_df),
            'high_confidence_constraints': len(high_confidence_df),
            'filter_efficiency_percent': len(filtered_df) / len(self.df) * 100 if self.df is not None else 0,
            'confidence_statistics': {
                'mean': float(filtered_df['confidence_score'].mean()),
                'median': float(filtered_df['confidence_score'].median()),
                'std': float(filtered_df['confidence_score'].std()),
                'min': float(filtered_df['confidence_score'].min()),
                'max': float(filtered_df['confidence_score'].max())
            },
            'methodology': {
                'min_confidence_threshold': 0.5,
                'regulatory_keywords_categories': len(self.regulatory_keywords),
                'non_regulatory_patterns': len(self.non_regulatory_patterns)
            },
            'output_columns': columns_to_keep if 'component_name' in original_columns else original_columns,
            'filtering_date': pd.Timestamp.now().isoformat()
        }
        
        # Save summary
        json_path = f"{output_path}_summary.json"
        with open(json_path, 'w', encoding='utf-8') as f:
            json.dump(summary, f, indent=2, ensure_ascii=False)
        print(f"💾 Summary saved to: {json_path}")
        
        # Create detailed text report
        txt_path = f"{output_path}_report.txt"
        with open(txt_path, 'w', encoding='utf-8') as f:
            f.write("GENUINE REGULATORY CONSTRAINTS EXTRACTION REPORT\n")
            f.write("=" * 60 + "\n\n")
            
            f.write(f"📊 EXECUTIVE SUMMARY:\n")
            f.write(f"  Original Constraints: {len(self.df) if self.df is not None else 0}\n")
            f.write(f"  Regulatory Constraints: {len(filtered_df)}\n")
            f.write(f"  High Confidence (≥0.8): {len(high_confidence_df)}\n")
            f.write(f"  Filter Efficiency: {len(filtered_df) / len(self.df) * 100:.1f}%\n")
            f.write(f"  Generated: {pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')}\n\n")
            
            f.write(f"📋 OUTPUT COLUMNS:\n")
            f.write(f"  Columns included: {', '.join(columns_to_keep if 'component_name' in original_columns else original_columns)}\n\n")
            
            f.write(f"🎯 FILTERING METHODOLOGY:\n")
            f.write(f"  • Regulatory keyword detection\n")
            f.write(f"  • Non-regulatory pattern exclusion\n")
            f.write(f"  • Confidence scoring based on linguistic features\n")
            f.write(f"  • Minimum confidence threshold: 0.5\n")
            f.write(f"  • Columns filtered: removed all columns after 'component_name'\n\n")
            
            f.write(f"📋 TOP 20 REGULATORY CONSTRAINTS:\n")
            f.write("-" * 60 + "\n")
            
            top_20 = filtered_df.head(20)
            for i, (idx, row) in enumerate(top_20.iterrows(), 1):
                f.write(f"\n{i:2d}. CONFIDENCE: {row['confidence_score']:.3f}\n")
                f.write(f"    Category: {row.get('category', 'N/A')}\n")
                f.write(f"    Description: {str(row.get('description', ''))[:300]}...\n")
                f.write(f"    Analysis: {row.get('analysis_reason', 'N/A')}\n")
                f.write(f"    Source: {str(row.get('source_section_number', ''))[:100]}...\n")
                if 'value' in row and pd.notna(row['value']):
                    f.write(f"    Value: {row['value']} {row.get('unit', '')}\n")
        
        print(f"💾 Detailed report saved to: {txt_path}")
        print(f"✅ Output files contain only columns up to 'component_name'")

print("✅ RegulationFilter class defined successfully!")

✅ RegulationFilter class defined successfully!


In [9]:
# Specify input and output paths here
input_csv_path = "/Users/li/Library/CloudStorage/OneDrive-UniversityofMaryland/PhD research/Python/LLM/New_documents_processed/all_processed_constraints.csv"
output_base_path = "/Users/li/Library/CloudStorage/OneDrive-UniversityofMaryland/PhD research/Python/LLM/New_documents_regulations/genuine_regulations"

In [10]:
# Initialize the regulation filter
reg_filter = RegulationFilter()

# Use the input and output paths specified above
if reg_filter.load_data(input_csv_path):
    print("\n🔍 Starting regulation filtering process...")
    
    # Filter for genuine regulatory constraints
    filtered_regulations = reg_filter.filter_regulations(min_confidence=0.5)
    
    if not filtered_regulations.empty:
        # Analyze the results
        reg_filter.analyze_results(filtered_regulations)
        
        # Save the filtered regulations using the specified output path
        reg_filter.save_filtered_regulations(filtered_regulations, output_base_path)
        
        # Also try higher confidence threshold with manual output path specification
        print(f"\n--- FILTERING WITH HIGHER CONFIDENCE (≥0.7) ---")
        high_confidence_regs = reg_filter.filter_regulations(min_confidence=0.7)
        
        if not high_confidence_regs.empty:
            print(f"✓ High-confidence regulations: {len(high_confidence_regs)}")
            reg_filter.save_filtered_regulations(
                high_confidence_regs,
                output_base_path.replace("genuine_regulations", "strict_regulations/strict_regulations")
            )
        
        print(f"\n🎉 Regulation filtering complete!")
        print(f"📁 Check the Filtered_regulations folder for results")
        
    else:
        print("❌ No regulatory constraints found meeting the criteria")
else:
    print("❌ Failed to load data")

✓ Loaded 1022 rows from /Users/li/Library/CloudStorage/OneDrive-UniversityofMaryland/PhD research/Python/LLM/New_documents_processed/all_processed_constraints.csv
  Columns: ['model_type', 'model_name', 'document_id', 'constraint_id', 'category', 'description', 'value', 'unit', 'source_section_number', 'geographic_scope', 'context_quote', 'component_name', 'value2', 'unit2']

🔍 Starting regulation filtering process...
🔍 Analyzing 1022 constraints for regulatory content...
  Processing row 0/1022...
  Processing row 100/1022...
  Processing row 200/1022...
  Processing row 300/1022...
  Processing row 400/1022...
  Processing row 500/1022...
  Processing row 600/1022...
  Processing row 700/1022...
  Processing row 800/1022...
  Processing row 900/1022...
  Processing row 1000/1022...
✓ Analysis complete!
  Total constraints analyzed: 1022
  Regulatory constraints found: 288
  Filter efficiency: 28.2%
  Confidence range: 0.500 - 1.000

📊 REGULATORY CONSTRAINT ANALYSIS

📈 CONFIDENCE SCOR

OSError: Cannot save file into a non-existent directory: '/Users/li/Library/CloudStorage/OneDrive-UniversityofMaryland/PhD research/Python/LLM/New_documents_regulations/strict_regulations'

In [ ]:
# Test the filtering system with some example texts
print("🧪 TESTING REGULATION FILTER WITH EXAMPLES")
print("="*50)

test_cases = [
    # Non-regulatory (should be filtered out)
    "The SRWEC would be one 320-kilovolt (kV) DC export cable bundle",
    "Wind power density 2 225 W/m [2] 200/250 W/m [2]",
    "The size of 140 m is about 10% larger than the rotor diameter, which is 126 m",
    "The project consists of 15 wind turbines",
    "The water depth in the area ranges from 20 to 30 meters",
    
    # Regulatory (should be kept)
    "Turbine foundations must not disturb marine protected areas within a 5 km radius",
    "An offshore wind measurement campaign at minimum should include measurements of horizontal wind speed and wind direction from blade tip to at least hub height",
    "The use of cable protection measures must not exceed 10 percent of the total export and inter-array cable length",
    "Construction activities shall not commence within 1 nautical mile of critical habitat during breeding season",
    "Noise levels during pile driving operations must not exceed 160 dB at 750 meters from the source"
]

reg_filter_test = RegulationFilter()

print("\nTest Results:")
print("-" * 50)

for i, text in enumerate(test_cases, 1):
    is_reg, reason, score = reg_filter_test.is_regulatory_constraint(text)
    status = "✅ REGULATORY" if is_reg else "❌ NON-REGULATORY"
    
    print(f"\n{i:2d}. {status} (Score: {score:.3f})")
    print(f"    Text: {text[:80]}...")
    print(f"    Reason: {reason}")

print(f"\n🎯 Filter appears to correctly identify regulatory vs non-regulatory content!")

🧪 TESTING REGULATION FILTER WITH EXAMPLES

Test Results:
--------------------------------------------------

 1. ❌ NON-REGULATORY (Score: 0.000)
    Text: The SRWEC would be one 320-kilovolt (kV) DC export cable bundle...
    Reason: Contains 1 regulatory context(s); Contains 1 descriptive phrase(s) (penalty)

 2. ❌ NON-REGULATORY (Score: 0.100)
    Text: Wind power density 2 225 W/m [2] 200/250 W/m [2]...
    Reason: Matches non-regulatory pattern: ^(?:The\s+)?(?:wind\s+power\s+density|power\s+dens...

 3. ❌ NON-REGULATORY (Score: 0.100)
    Text: The size of 140 m is about 10% larger than the rotor diameter, which is 126 m...
    Reason: Matches non-regulatory pattern: about\s+\d+%\s+(?:larger|smaller|more|less)...

 4. ❌ NON-REGULATORY (Score: 0.000)
    Text: The project consists of 15 wind turbines...
    Reason: Contains 1 regulatory context(s); Contains 1 descriptive phrase(s) (penalty)

 5. ❌ NON-REGULATORY (Score: 0.150)
    Text: The water depth in the area ranges from 20 to 

In [ ]:
# Create a comparison between original and filtered datasets
if 'reg_filter' in locals() and reg_filter.df is not None and reg_filter.filtered_regulations is not None:
    print("📊 DETAILED COMPARISON: ORIGINAL vs FILTERED")
    print("="*60)
    
    original_df = reg_filter.df
    filtered_df = reg_filter.filtered_regulations
    
    print(f"\n📈 QUANTITATIVE COMPARISON:")
    print(f"  Original constraints: {len(original_df):,}")
    print(f"  Filtered regulations: {len(filtered_df):,}")
    print(f"  Reduction: {len(original_df) - len(filtered_df):,} ({(1 - len(filtered_df)/len(original_df))*100:.1f}%)")
    print(f"  Retention rate: {len(filtered_df)/len(original_df)*100:.1f}%")
    
    # Analyze what was filtered out
    if 'is_regulatory' in original_df.columns:
        non_regulatory = original_df[original_df['is_regulatory'] == False]
        print(f"\n❌ FILTERED OUT ({len(non_regulatory)} items):")
        print(f"  Mean confidence score: {non_regulatory['confidence_score'].mean():.3f}")
        
        # Show examples of filtered out content
        print(f"\n  Examples of filtered out content:")
        for i, (idx, row) in enumerate(non_regulatory.sample(min(5, len(non_regulatory))).iterrows(), 1):
            print(f"    {i}. Score: {row['confidence_score']:.3f}")
            print(f"       Text: {str(row['requirement'])[:100]}...")
            print(f"       Reason: {row['analysis_reason'][:80]}...")
    
    # Analyze constraint types in filtered data
    if 'constraint_type' in filtered_df.columns:
        print(f"\n📋 CONSTRAINT TYPES IN FILTERED DATA:")
        type_distribution = filtered_df['constraint_type'].value_counts()
        total_filtered = len(filtered_df)
        
        for constraint_type, count in type_distribution.head(15).items():
            percentage = (count / total_filtered) * 100
            print(f"  {constraint_type}: {count} ({percentage:.1f}%)")
    
    # Analyze confidence distribution
    print(f"\n📊 CONFIDENCE SCORE DISTRIBUTION:")
    confidence_bands = [
        (0.9, 1.0, "Very High"),
        (0.8, 0.9, "High"),
        (0.7, 0.8, "Medium-High"),
        (0.6, 0.7, "Medium"),
        (0.5, 0.6, "Low-Medium")
    ]
    
    for min_conf, max_conf, label in confidence_bands:
        count = len(filtered_df[(filtered_df['confidence_score'] >= min_conf) & 
                                (filtered_df['confidence_score'] < max_conf)])
        if min_conf == 0.9:  # Handle the top band
            count = len(filtered_df[filtered_df['confidence_score'] >= min_conf])
        
        percentage = (count / len(filtered_df)) * 100 if len(filtered_df) > 0 else 0
        print(f"  {label} ({min_conf:.1f}-{max_conf:.1f}): {count} ({percentage:.1f}%)")
    
    print(f"\n✅ Analysis complete - genuine regulations successfully extracted!")
else:
    print("❌ No filtering results available. Run the filtering process first.")

📊 DETAILED COMPARISON: ORIGINAL vs FILTERED

📈 QUANTITATIVE COMPARISON:
  Original constraints: 1,022
  Filtered regulations: 181
  Reduction: 841 (82.3%)
  Retention rate: 17.7%

📊 CONFIDENCE SCORE DISTRIBUTION:
  Very High (0.9-1.0): 74 (40.9%)
  High (0.8-0.9): 53 (29.3%)
  Medium-High (0.7-0.8): 54 (29.8%)
  Medium (0.6-0.7): 0 (0.0%)
  Low-Medium (0.5-0.6): 0 (0.0%)

✅ Analysis complete - genuine regulations successfully extracted!
